In [ ]:
import os
import numpy as np
import nibabel as nib
import torch
import matplotlib.pyplot as plt
from skimage.measure import marching_cubes
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from monai.transforms import Resize
from monai.data import MetaTensor
from monai.utils import ensure_tuple_rep


# Utility: Save 3D surface plot
def plot_3d(volume, title, filename):
    verts, faces, _, _ = marching_cubes(volume, 0.5)
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    mesh = Poly3DCollection(verts[faces], alpha=0.70)
    mesh.set_edgecolor('k')
    ax.add_collection3d(mesh)
    ax.set_xlim(0, volume.shape[2])
    ax.set_ylim(0, volume.shape[1])
    ax.set_zlim(0, volume.shape[0])
    ax.set_title(title)
    plt.savefig(filename)
    plt.close()

# STEP 1: Load raw CT
raw_ct_path = "data/ct.nii.gz"
raw_ct = nib.load(raw_ct_path).get_fdata()
plot_3d(raw_ct, "Raw CT", "1_raw_ct.png")

# STEP 2: Run TotalSegmentator (assumes CLI installed)
os.system(f"TotalSegmentator -i {raw_ct_path} -o output_seg --fast")
heart_mask = nib.load("output_seg/heart.nii.gz").get_fdata()
plot_3d(heart_mask, "Segmented Heart", "2_heart_mask.png")

# STEP 3: Remove empty slices
non_empty_slices = np.any(heart_mask, axis=(1, 2))
trimmed_mask = heart_mask[non_empty_slices]
plot_3d(trimmed_mask, "Trimmed Heart Mask", "3_trimmed_mask.png")

# STEP 4: Trim original CT using mask bounds
trimmed_ct = raw_ct[non_empty_slices]
plot_3d(trimmed_ct, "Trimmed CT", "4_trimmed_ct.png")

# STEP 5: Resize heart mask
target_shape = (64, 128, 128)
mask_tensor = torch.tensor(trimmed_mask[None, None], dtype=torch.float32)
resized_mask = Resize(spatial_size=target_shape)(mask_tensor).squeeze().numpy()
plot_3d(resized_mask, "Resized Mask", "5_resized_mask.png")

# STEP 6: Resize original CT
ct_tensor = torch.tensor(trimmed_ct[None, None], dtype=torch.float32)
resized_ct = Resize(spatial_size=target_shape)(ct_tensor).squeeze().numpy()
plot_3d(resized_ct, "Resized CT", "6_resized_ct.png")

# STEP 7: Run Grad-CAM (assumes model + hook set up)
input_tensor = torch.tensor(resized_ct[None, None]).float().to("cuda")  # [B,C,D,H,W]
input_tensor.requires_grad = True
output = model(input_tensor)
output[0].backward()
gradcam = model.get_last_cam().detach().cpu().squeeze().numpy()  # e.g. [D,H,W]
plt.imshow(gradcam[gradcam.shape[0]//2], cmap="jet")
plt.title("Grad-CAM Heatmap")
plt.colorbar()
plt.savefig("7_heatmap.png")
plt.close()

# STEP 8: Overlay heatmap on resized mask
overlay_mask = resized_mask * gradcam
plt.imshow(overlay_mask[overlay_mask.shape[0]//2], cmap="hot")
plt.title("Heatmap on Mask")
plt.colorbar()
plt.savefig("8_overlay_mask.png")
plt.close()

# STEP 9: Overlay heatmap on resized CT
overlay_ct = resized_ct * gradcam
plt.imshow(overlay_ct[overlay_ct.shape[0]//2], cmap="hot")
plt.title("Heatmap on CT")
plt.colorbar()
plt.savefig("9_overlay_ct.png")
plt.close()
